# Sentiment Analysis: Synthetic Data Generation for Google Colab

## Fast Multi-Model Conversation Generation Pipeline - GPU Optimized

This notebook generates synthetic conversations using **FREE** open-source LLMs (Llama 2, Mistral, Neural Chat, Phi) via Ollama, optimized for Google Colab and Kaggle with GPU acceleration.

**Speed Benefits:**
- ✅ **GPU-Accelerated** - Uses Colab's free GPUs for faster inference
- ✅ Parallel model execution
- ✅ Automatic result download
- ✅ Production-ready sentiment analysis

**Cost:** $0.00 (completely free with GPU acceleration)

In [ ]:
import os
import subprocess
import sys

# Detect environment
IS_COLAB = 'google.colab' in sys.modules
IS_KAGGLE = os.path.exists('/kaggle/working')

print(f"Environment: {'Google Colab' if IS_COLAB else 'Kaggle' if IS_KAGGLE else 'Local'}")

# GPU Detection and Configuration
try:
    import torch
    USE_GPU = torch.cuda.is_available()
    GPU_COUNT = torch.cuda.device_count()
    GPU_NAME = torch.cuda.get_device_name(0) if USE_GPU else "None"
    CUDA_VERSION = torch.version.cuda
except ImportError:
    USE_GPU = False
    GPU_COUNT = 0
    GPU_NAME = "N/A"
    CUDA_VERSION = "N/A"

print(f"\n🔧 GPU Configuration:")
print(f"   CUDA Available: {'✅ YES' if USE_GPU else '❌ NO'}")
if USE_GPU:
    print(f"   GPU Count: {GPU_COUNT}")
    print(f"   GPU Device: {GPU_NAME}")
    print(f"   CUDA Version: {CUDA_VERSION}")
    print(f"\n✨ GPU acceleration ENABLED for faster processing!")
else:
    print(f"   ⚠️  Running on CPU")

# Install dependencies
print("\nInstalling dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests", "jsonlines", "tqdm", "numpy", "pandas"])
print("✅ Dependencies installed")

## Step 1: Install Ollama and Download Models

This will install Ollama and download the 4 free LLM models needed for conversation generation.

In [ ]:
import subprocess
import time
import threading
import requests
import os

print("📦 Installing Ollama...")
ollama_executable_path = "/usr/local/bin/ollama"  # Expected installation path

# Configure GPU environment variables
if USE_GPU:
    os.environ['CUDA_VISIBLE_DEVICES'] = '0'
    os.environ['OLLAMA_GPU_LAYERS'] = '99'
    os.environ['OLLAMA_NUM_GPU'] = '1'
    print("🔧 GPU environment configured for Ollama")
else:
    print("⚠️  GPU not detected - will run on CPU")

try:
    # Install zstd dependency as requested by the Ollama installer
    print("Installing zstd dependency...")
    subprocess.run(
        ["sudo", "apt-get", "update"],
        capture_output=True, text=True, check=True
    )
    subprocess.run(
        ["sudo", "apt-get", "install", "-y", "zstd"],
        capture_output=True, text=True, check=True
    )
    print("✅ zstd installed successfully.")

    # Correctly execute the Ollama installation script by piping curl output to sh
    print("Running Ollama installation script...")
    install_command = "curl -fsSL https://ollama.ai/install.sh | sh"
    install_result = subprocess.run(
        install_command,
        shell=True,
        capture_output=True,
        text=True,
        check=True
    )
    print("✅ Ollama installation script executed successfully.")
    if install_result.stdout:
        print("Installer Output (stdout):\n", install_result.stdout)
    if install_result.stderr:
        print("Installer Output (stderr):\n", install_result.stderr)

    # Verify if the ollama executable exists after installation
    if not os.path.exists(ollama_executable_path):
        raise FileNotFoundError(f"Ollama executable not found at {ollama_executable_path} after installation.")
    print(f"✅ Ollama executable verified at: {ollama_executable_path}")

except subprocess.CalledProcessError as e:
    print(f"❌ A subprocess command failed with return code {e.returncode}.")
    print("Command:", e.cmd)
    print("Stdout:", e.stdout)
    print("Stderr:", e.stderr)
    print("Please check the output for errors during installation or dependency setup.")
    raise
except FileNotFoundError as e:
    print(f"❌ Error: {e}")
    raise
except Exception as e:
    print(f"❌ An unexpected error occurred during Ollama installation: {e}")
    raise


# Start Ollama in background with GPU support
print("\n🚀 Starting Ollama service...")
def run_ollama():
    env = os.environ.copy()
    if USE_GPU:
        env['CUDA_VISIBLE_DEVICES'] = '0'
        env['OLLAMA_GPU_LAYERS'] = '99'
        env['OLLAMA_NUM_GPU'] = '1'
    
    subprocess.Popen(
        [ollama_executable_path, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        env=env
    )

# Start Ollama in thread
ollama_thread = threading.Thread(target=run_ollama, daemon=True)
ollama_thread.start()

# Wait for Ollama to start
ollama_started = False
for i in range(30):
    try:
        response = requests.get("http://localhost:11434/api/tags", timeout=2)
        if response.status_code == 200:
            print("✅ Ollama is running")
            if USE_GPU:
                print("   GPU acceleration enabled 🚀")
            ollama_started = True
            break
    except requests.exceptions.ConnectionError:
        pass
    except Exception as e:
        print(f"⚠️ Error checking Ollama status: {e}")
    time.sleep(1)
    if i % 5 == 0:
        print(f"⏳ Waiting for Ollama... ({i+1}s)")

if not ollama_started:
    print("❌ Ollama service did not start within the expected time. Please check logs for errors.")
    raise RuntimeError("Ollama service failed to start.")


print("\n📥 Downloading models (this takes ~10-20 minutes on first run)...")
models = ["llama2", "mistral", "neural-chat", "phi"]

for model in models:
    print(f"\n⬇️  Downloading {model}...")
    result = subprocess.run([ollama_executable_path, "pull", model],
                          capture_output=True, text=True)
    if "success" in result.stdout.lower() or "latest" in result.stdout.lower():
        print(f"✅ {model} ready")
    else:
        print(f"📥 {model} downloading in background (or failed, check stderr)...")
        if result.stdout:
            print(f"Stdout for {model}:\n", result.stdout)
        if result.stderr:
            print(f"Stderr for {model}:\n", result.stderr)

print("\n✅ All models ready for generation!")
if USE_GPU:
    print("   Running with GPU acceleration ⚡")

## Step 2: Define Multi-Model Conversation Generator

Fast, efficient conversation generation using FREE open-source models.

In [ ]:
import requests
import json
import time
from datetime import datetime
from typing import Dict, List, Tuple
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class OllamaConversationGenerator:
    def __init__(self, base_url="http://localhost:11434", timeout=180):
        self.base_url = base_url
        self.timeout = timeout
        
        self.TOPICS = [
            "artificial intelligence and machine learning",
            "climate change and sustainability",
            "space exploration and astronomy",
            "mental health and wellness",
            "cryptocurrency and blockchain",
            "remote work and productivity",
            "quantum computing",
            "renewable energy",
            "social media impact",
            "pandemic response strategies",
            "education technology",
            "autonomous vehicles",
        ]
        
        self.PROMPTS = {
            "user_base": "Let's discuss {topic}. What are your thoughts?",
            "continuation": "Based on that, I think...",
        }
    
    def call_ollama(self, model: str, prompt: str) -> Tuple[str, Dict]:
        """Call Ollama API for text generation"""
        try:
            response = requests.post(
                f"{self.base_url}/api/generate",
                json={"model": model, "prompt": prompt, "stream": False},
                timeout=self.timeout,
            )
            response.raise_for_status()
            result = response.json()
            return result.get("response", ""), {"status": "success"}
        except Exception as e:
            logger.warning(f"Ollama call failed: {e}")
            return "", {"status": "error", "error": str(e)}
    
    def generate_conversation(self, model_a: str, model_b: str, topic: str, turns: int = 3) -> Dict:
        """Generate a conversation between two models"""
        messages = []
        
        # Model A starts
        prompt_a = f"You are an expert discussing: {topic}. Be concise (1-2 sentences)."
        response_a, _ = self.call_ollama(model_a, prompt_a)
        
        if not response_a:
            return None
        
        messages.append({
            "turn": 1,
            "speaker": model_a,
            "message": response_a.strip()[:200],
        })
        
        time.sleep(0.3)  # Rate limiting
        
        # Model B responds
        prompt_b = f"Respond to this: {messages[-1]['message']}\n\nStay concise (1-2 sentences)."
        response_b, _ = self.call_ollama(model_b, prompt_b)
        
        if response_b:
            messages.append({
                "turn": 1,
                "speaker": model_b,
                "message": response_b.strip()[:200],
            })
        
        return {
            "timestamp": datetime.now().isoformat(),
            "model_a": model_a,
            "model_b": model_b,
            "topic": topic,
            "messages": messages,
            "turn_count": len(messages),
        }

# Initialize generator
generator = OllamaConversationGenerator()
print("✅ Conversation generator initialized")

## Step 3: Configure Generation Parameters

Adjust these parameters to control generation speed and dataset size.

In [ ]:
# ⚙️ CONFIGURATION - Modify these for faster/slower generation
CONFIG = {
    # Model pairs for conversation generation
    "model_pairs": [
        ("llama2", "mistral"),
        ("mistral", "phi"),
        ("phi", "llama2"),
        ("llama2", "phi"),
        ("mistral", "neural-chat"),
        ("neural-chat", "llama2"),
    ],
    
    # Number of conversations per pair
    # Total conversations: 6 pairs × 500 = 3000
    "conversations_per_pair": 500,
    
    # Topics to discuss in conversations
    "topics": [
        "artificial intelligence",
        "climate change",
        "space exploration",
        "mental health",
        "cryptocurrency",
        "remote work",
        "renewable energy",
        "autonomous vehicles",
        "online privacy",
        "education technology",
    ],
    
    # Output configuration
    "output_dir": "/content/synthetic_data" if IS_COLAB else "/kaggle/working/synthetic_data",
    "output_file": "conversations.jsonl",
}

# Create output directory
import os
os.makedirs(CONFIG["output_dir"], exist_ok=True)

print("⚙️ Configuration:")
print(f"  Model pairs: {len(CONFIG['model_pairs'])}")
print(f"  Conversations per pair: {CONFIG['conversations_per_pair']}")
print(f"  Total conversations: {len(CONFIG['model_pairs']) * CONFIG['conversations_per_pair']}")
print(f"  Topics: {len(CONFIG['topics'])}")
print(f"  Output: {CONFIG['output_dir']}")

## Step 4: Generate Synthetic Conversations

This will generate realistic conversations between model pairs. **Runtime: 5-20 minutes depending on settings.**

In [ ]:
import time
import random
from tqdm import tqdm

print("🚀 Starting conversation generation...\n")

output_path = os.path.join(CONFIG["output_dir"], CONFIG["output_file"])
conversation_count = 0
start_time = time.time()

# Total conversations to generate
total_conversations = len(CONFIG['model_pairs']) * CONFIG['conversations_per_pair']

with open(output_path, 'w') as f:
    for pair_idx, (model_a, model_b) in enumerate(CONFIG['model_pairs']):
        print(f"\n📍 Model pair {pair_idx + 1}/{len(CONFIG['model_pairs'])}: {model_a} ↔ {model_b}")
        
        pair_start = time.time()
        successful = 0
        
        for conv_idx in tqdm(range(CONFIG['conversations_per_pair']), desc=f"Generating {model_a}-{model_b}"):
            topic = random.choice(CONFIG['topics'])
            
            try:
                conversation = generator.generate_conversation(model_a, model_b, topic, turns=2)
                
                if conversation and len(conversation.get('messages', [])) > 0:
                    f.write(json.dumps(conversation) + '\n')
                    conversation_count += 1
                    successful += 1
            except Exception as e:
                logger.warning(f"Generation error: {e}")
                continue
        
        pair_time = time.time() - pair_start
        print(f"✅ Generated {successful} conversations in {pair_time:.1f}s")

# Summary
elapsed_time = time.time() - start_time
print(f"\n{'='*60}")
print(f"🎉 GENERATION COMPLETE!")
print(f"{'='*60}")
print(f"✅ Total conversations: {conversation_count}")
print(f"⏱️  Total time: {elapsed_time:.1f}s ({elapsed_time/60:.1f} minutes)")
print(f"📊 Rate: {conversation_count/(elapsed_time/60):.1f} conversations/minute")
print(f"💾 Output file: {output_path}")
print(f"📦 File size: {os.path.getsize(output_path) / 1024 / 1024:.2f} MB")

## Step 5: Run Sentiment Analysis

Fast sentiment analysis using VADER (no model download required).

In [ ]:
import subprocess
import sys

# Install VADER sentiment analyzer
print("📥 Installing VADER sentiment analyzer...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "nltk"])

import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# Download required VADER data
try:
    nltk.data.find('vader_lexicon')
except LookupError:
    nltk.download('vader_lexicon', quiet=True)

sia = SentimentIntensityAnalyzer()
print("✅ VADER initialized")

# Run sentiment analysis on generated conversations
print("\n🔍 Running sentiment analysis on generated conversations...\n")

analysis_output = os.path.join(CONFIG["output_dir"], "analysis_results.jsonl")
sentiment_scores = []

with open(output_path, 'r') as infile, open(analysis_output, 'w') as outfile:
    for line in tqdm(infile, desc="Analyzing sentiment", total=conversation_count):
        conversation = json.loads(line)
        
        # Analyze each message
        for msg in conversation.get('messages', []):
            text = msg.get('message', '')
            scores = sia.polarity_scores(text)
            
            analysis = {
                "timestamp": conversation.get('timestamp'),
                "model": msg.get('speaker'),
                "message": text,
                "sentiment": {
                    "positive": round(scores['pos'], 3),
                    "negative": round(scores['neg'], 3),
                    "neutral": round(scores['neu'], 3),
                    "compound": round(scores['compound'], 3),
                },
                "sentiment_label": "positive" if scores['compound'] > 0.05 else ("negative" if scores['compound'] < -0.05 else "neutral"),
            }
            sentiment_scores.append(analysis['sentiment_label'])
            outfile.write(json.dumps(analysis) + '\n')

# Summary statistics
print(f"\n{'='*60}")
print(f"📊 SENTIMENT ANALYSIS SUMMARY")
print(f"{'='*60}")
print(f"Total messages analyzed: {len(sentiment_scores)}")
print(f"Positive: {sentiment_scores.count('positive')} ({sentiment_scores.count('positive')/len(sentiment_scores)*100:.1f}%)")
print(f"Negative: {sentiment_scores.count('negative')} ({sentiment_scores.count('negative')/len(sentiment_scores)*100:.1f}%)")
print(f"Neutral:  {sentiment_scores.count('neutral')} ({sentiment_scores.count('neutral')/len(sentiment_scores)*100:.1f}%)")

## Step 6: Download Results

Export the generated synthetic data and analysis results.

In [ ]:
if IS_COLAB:
    from google.colab import files
    
    print("📥 Preparing files for download...\n")
    
    # List files to download
    files_to_download = [
        output_path,
        analysis_output,
    ]
    
    print("📦 Downloading files:")
    for file_path in files_to_download:
        if os.path.exists(file_path):
            file_size = os.path.getsize(file_path) / 1024 / 1024
            print(f"  • {os.path.basename(file_path)} ({file_size:.2f} MB)")
    
    print("\n⏳ Starting download... Check your Downloads folder")
    
    for file_path in files_to_download:
        if os.path.exists(file_path):
            files.download(file_path)
    
    print("\n✅ Download complete!")
    
elif IS_KAGGLE:
    print("📊 Files saved to /kaggle/working/")
    print(f"  • conversations.jsonl")
    print(f"  • analysis_results.jsonl")
    print("\nYou can download these from the Kaggle output panel.")
    
else:
    print("📊 Files saved locally:")
    print(f"  • {output_path}")
    print(f"  • {analysis_output}")

## Step 7: Performance Benchmarking

Compare execution speed and efficiency of Colab vs local.

In [ ]:
import psutil
import platform

print("=" * 70)
print("🖥️  PERFORMANCE BENCHMARK")
print("=" * 70)

# System info
print(f"\n📊 System Information:")
print(f"  Platform: {platform.platform()}")
print(f"  Environment: {'Google Colab' if IS_COLAB else 'Kaggle' if IS_KAGGLE else 'Local'}")

# CPU info
print(f"\n💻 Processor:")
print(f"  CPU Count: {psutil.cpu_count()}")
print(f"  CPU Freq: {psutil.cpu_freq().current:.0f} MHz")

# Memory info
mem = psutil.virtual_memory()
print(f"\n🧠 Memory:")
print(f"  Total: {mem.total / 1024 / 1024 / 1024:.1f} GB")
print(f"  Available: {mem.available / 1024 / 1024 / 1024:.1f} GB")

# Generation metrics
print(f"\n⚡ Generation Performance:")
print(f"  Total conversations: {conversation_count}")
print(f"  Total time: {elapsed_time:.1f}s ({elapsed_time/60:.1f} min)")
print(f"  Rate: {conversation_count / (elapsed_time/60):.1f} conversations/min")
print(f"  Time per conversation: {elapsed_time / conversation_count:.1f}s")

# Data size
total_size = os.path.getsize(output_path) + os.path.getsize(analysis_output)
print(f"\n💾 Data Size:")
print(f"  Conversations file: {os.path.getsize(output_path) / 1024 / 1024:.2f} MB")
print(f"  Analysis file: {os.path.getsize(analysis_output) / 1024 / 1024:.2f} MB")
print(f"  Total: {total_size / 1024 / 1024:.2f} MB")

# Cost comparison
print(f"\n💰 Cost Analysis:")
print(f"  Colab/Kaggle: $0.00 (FREE! 🎉)")
print(f"  vs OpenAI API @ $0.01/1K tokens: ~${conversation_count * 0.5 * 0.01 / 1000:.2f}")
print(f"  vs Anthropic @ $0.008/1K tokens: ~${conversation_count * 0.5 * 0.008 / 1000:.2f}")

print(f"\n" + "=" * 70)
print(f"✅ All tasks completed successfully!")
print(f"=" * 70)